In [2]:
# Load environment variables and models
import os
from dotenv import load_dotenv

# Load API key
load_dotenv(dotenv_path="C:/Support-Ticket-Classifier-with-RAG/key.env")
groq_token = os.getenv("GROQ_API_KEY")

# Load Groq LLM
from langchain_groq import ChatGroq
groq_model = ChatGroq(api_key=groq_token,model="llama-3.1-8b-instant")

# Load HuggingFace Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embeds = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Support-Ticket-Classifier-with-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3509.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
import os
# Function to save knowledge base
def save_knowledge_base(data, file_path):

    directory = os.path.dirname(file_path)

    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory '{directory}' created.")
    else:
        print(f"Directory '{directory}' already exists.")

    with open(file_path, "w", encoding="utf-8") as file:
        for entry in data:
            file.write(entry + "\n\n")

    print(f"Knowledge base saved to {file_path}")

# Knowledge Base
knowledge_base = [

"""
Login Issues

Common Problems:
- Incorrect password
- Account locked
- Password reset failure
- Verification code issues
- Login session expired

Recommended Solutions:
- Reset the password
- Clear browser cache
- Check email spam folder
- Verify username and email
- Contact support if account remains locked
""",

"""
Application Functionality Issues

Common Problems:
- App crashes
- Upload failures
- Buttons not responding
- Notifications not working
- Features unavailable

Recommended Solutions:
- Restart the application
- Update the app
- Clear cache
- Reinstall the application
- Check device compatibility
""",

"""
Billing and Payment Issues

Common Problems:
- Duplicate charges
- Failed subscription
- Refund delays
- Payment declined
- Missing invoices

Recommended Solutions:
- Verify payment history
- Retry payment method
- Contact bank support
- Wait for payment confirmation
- Contact billing support
""",

"""
Account Management Issues

Common Problems:
- Cannot update profile
- Profile picture upload failure
- Privacy settings not saving
- Social account linking issues

Recommended Solutions:
- Refresh the page
- Verify entered information
- Use supported image formats
- Re-login and try again
""",

"""
Performance Issues

Common Problems:
- Slow loading
- Video buffering
- App freezing
- High memory usage
- Delayed responses

Recommended Solutions:
- Restart the app
- Check internet connection
- Close background applications
- Update the application
- Restart the device
"""
]

# Save file
kb_file_path = "C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base2.txt"
save_knowledge_base(knowledge_base, kb_file_path)

Directory 'C:/Support-Ticket-Classifier-with-RAG/data' already exists.
Knowledge base saved to C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base2.txt


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Function to load and split documents
def load_and_split_docs(file_path, chunk_size=500, chunk_overlap=50):

    loader = TextLoader(file_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

# Load chunks
kb_chunks = load_and_split_docs(kb_file_path)

# Preview chunks
kb_chunks

[Document(metadata={'source': 'C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base2.txt'}, page_content='Login Issues\n\nCommon Problems:\n- Incorrect password\n- Account locked\n- Password reset failure\n- Verification code issues\n- Login session expired\n\nRecommended Solutions:\n- Reset the password\n- Clear browser cache\n- Check email spam folder\n- Verify username and email\n- Contact support if account remains locked\n\n\n\nApplication Functionality Issues\n\nCommon Problems:\n- App crashes\n- Upload failures\n- Buttons not responding\n- Notifications not working\n- Features unavailable'),
 Document(metadata={'source': 'C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base2.txt'}, page_content='Recommended Solutions:\n- Restart the application\n- Update the app\n- Clear cache\n- Reinstall the application\n- Check device compatibility\n\n\n\nBilling and Payment Issues\n\nCommon Problems:\n- Duplicate charges\n- Failed subscription\n- Refund delays\n- Payment declined\n

In [5]:
from langchain_community.vectorstores import FAISS

# Function to create vector store
def create_vector_store(documents, embeddings_model):

    return FAISS.from_documents(documents=documents,embedding=embeddings_model)

# Create vector database
kb_vectorstore = create_vector_store(kb_chunks, embeds)

In [6]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt
guidelines_prompt = (
"""
You are an AI technical support assistant.

Use the retrieved background information below to generate
a short and practical solution for the user's issue.

Background Information:
{context}

Instructions:
1. Understand the support issue carefully.
2. Generate ONLY a short practical solution.
3. Keep the response clear and concise.
4. Maximum 2-3 lines.
5. Do not classify the issue.
6. Do not explain your reasoning.
7. If unsure, respond:
Cannot determine a proper solution.

Output Format:
Solution: <short solution>
"""
)

# Prompt template
qa_template = ChatPromptTemplate.from_messages(
    [
        ("system", guidelines_prompt),
        ("human", "{input}")
    ]
)

In [7]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Create QA chain
qa_chain = create_stuff_documents_chain(groq_model,qa_template)

# Create retriever
retriever = kb_vectorstore.as_retriever(k=3)

# Create RAG chain
rag_chain = create_retrieval_chain(retriever,qa_chain)

In [8]:
# Function to generate solutions
def generate_ticket_solutions(tickets, rag_chain):

    solved_tickets = []

    for ticket in tickets:

        ticket_text = ticket['text']
        response = rag_chain.invoke({ "input": ticket_text})

        solved_tickets.append({"ticket": ticket_text, "solution": response['answer']})

    return solved_tickets


# Display results
def display_ticket_solutions(solved_tickets):

    print("\n" + "=" * 70)
    print("💡 SUPPORT TICKET SOLUTIONS")
    print("=" * 70 + "\n")

    for i, ticket_info in enumerate(solved_tickets, 1):
        print(f"🎫 Ticket {i}")
        print("-" * 70)
        print("📝 Issue:")
        print(ticket_info['ticket'])
        print("\n💡 Solution:")
        print(ticket_info['solution'])
        print("\n" + "=" * 70 + "\n")

# Example tickets
support_tickets = [

    {
        "text": "I keep getting 'account locked' message even though my password is correct."
    },

    {
        "text": "After updating the app, it closes immediately when I open it."
    },

    {
        "text": "My credit card was charged but my subscription still shows as inactive."
    },

    {
        "text": "I want to unlink my Facebook account but the button is grayed out."
    },

    {
        "text": "hello."
    }
]

# Generate solutions
solved_tickets = generate_ticket_solutions(
    support_tickets,
    rag_chain
)

# Display output
display_ticket_solutions(solved_tickets)


💡 SUPPORT TICKET SOLUTIONS

🎫 Ticket 1
----------------------------------------------------------------------
📝 Issue:
I keep getting 'account locked' message even though my password is correct.

💡 Solution:
Solution: Check your email spam folder for account lock verification emails, then follow the instructions to unlock your account.


🎫 Ticket 2
----------------------------------------------------------------------
📝 Issue:
After updating the app, it closes immediately when I open it.

💡 Solution:
Solution: Restart the app or your device to resolve the issue.


🎫 Ticket 3
----------------------------------------------------------------------
📝 Issue:
My credit card was charged but my subscription still shows as inactive.

💡 Solution:
Solution: Retry payment method or verify payment history with the billing support.


🎫 Ticket 4
----------------------------------------------------------------------
📝 Issue:
I want to unlink my Facebook account but the button is grayed out.

💡 Soluti